# EDA Bloque 3 — Percepción Emocional (NLP)

```
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 3: PERCEPCION EMOCIONAL (IA / NLP)
================================================================================
Datasets analizados (2):
  3.1 dreaddit-train.csv + dreaddit-test.csv     (Kaggle: payaldhokane)
      -> corpus textual real de Reddit con etiqueta de estres. Fuente NLP.
  3.2 Depression+Student+Dataset.csv             (GitHub: prajwalg26)
      -> tabular; complementa el perfil de malestar estudiantil. NO es NLP.

Para el corpus textual, el EDA incluye pasos especificos de texto:
  PASO 3-T: longitud de posts, balance de clases, distribucion por subreddit
  PASO 4-T: senal lexica (sentimiento, categorias LIWC) por clase
================================================================================
```

---
**Cómo leer este notebook.** Cada dataset se analiza con la misma plantilla de 5 pasos:

| Paso | Qué hace | Comandos clave |
|---|---|---|
| 1. Carga | Lee el CSV y muestra tamaño y variables | `pd.read_csv()` |
| 2. Calidad | Cuenta nulos y duplicados | `.isna()`, `.duplicated()` |
| 3. Univariante | Distribución de la variable de estrés | `.value_counts()`, `.describe()` |
| 4. Bivariante | Relación de cada variable con el estrés | `.corr()`, `.groupby()`, `pd.crosstab()` |
| 5. Hallazgos | Conclusiones del dataset | — |

Las gráficas se guardan automáticamente como PNG en la carpeta de figuras (`plt.savefig`).


**Configuración.** Importa las librerías (`pandas` para datos, `matplotlib`/`seaborn` para gráficas), define las rutas de entrada (`DATA`) y salida de figuras (`FIG`) y crea la carpeta de figuras si no existe (`os.makedirs`). `SEP` es solo una línea decorativa para separar secciones en la salida.

In [27]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ── Configuracion global ──────────────────────────────────────────────────────
BASE = r"../data/raw"
ALMC = r"../salidas/objetivo1"
DATA = os.path.join(BASE, "Bloque 3") + os.sep
FIG  = os.path.join(ALMC, "figurasObje1") + os.sep
os.makedirs(FIG, exist_ok=True)   # crea la carpeta de figuras1 si no existe

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

## DATASET 3.1 — DREADDIT (payaldhokane, train n=2838 / test n=715)

In [28]:
print(SEP); print("DATASET 3.1 - DREADDIT (payaldhokane)"); print(SEP)

DATASET 3.1 - DREADDIT (payaldhokane)


**Paso 1 – Carga e inventario.** `pd.read_csv()` carga el archivo en un DataFrame (tabla en memoria); `.shape` muestra filas × columnas.

In [29]:
# PASO 1: Carga --------------------------------------------------------------
tr = pd.read_csv(DATA + "3 Stress Analysis in Social Media Dataset/dreaddit-train.csv")
te = pd.read_csv(DATA + "3 Stress Analysis in Social Media Dataset/dreaddit-test.csv")
print(f"Train: {tr.shape} | Test: {te.shape}")
print("Columnas clave: text (post), label (0/1), subreddit, sentiment, "
      "107 categorias LIWC, indices DAL y legibilidad (116 columnas total)")

Train: (2838, 116) | Test: (715, 116)
Columnas clave: text (post), label (0/1), subreddit, sentiment, 107 categorias LIWC, indices DAL y legibilidad (116 columnas total)


**Paso 2 – Calidad.** `.isna().sum()` cuenta valores vacíos por columna y `.duplicated().sum()` cuenta filas repetidas.

In [30]:
# PASO 2: Calidad ------------------------------------------------------------
print("Nulos train:", tr.isna().sum().sum(), "| Nulos test:", te.isna().sum().sum())
print("Particion train/test ya definida por los autores: se respetara en el "
      "Objetivo 2 para comparabilidad con la literatura.")

Nulos train: 0 | Nulos test: 0
Particion train/test ya definida por los autores: se respetara en el Objetivo 2 para comparabilidad con la literatura.


**Paso 3-T – EDA textual.** Para texto no hay 'distribuciones' clásicas: se mide el balance de clases (estrés/no estrés), la longitud de los posts (`.str.len()`, `.str.split().str.len()` = caracteres y palabras) y de qué subreddit proviene cada post.

In [31]:
# PASO 3-T: EDA textual - balance, longitud, subreddits ------------------------
tr["text_len"] = tr["text"].str.len()
tr["n_words"] = tr["text"].str.split().str.len()
print("\nBalance de clases:\n", tr["label"].value_counts())
print("\nLongitud (palabras) por clase:\n",
      tr.groupby("label")["n_words"].describe()[["mean", "50%", "max"]].round(1))
print("\nPosts por subreddit:\n", tr["subreddit"].value_counts())

C:\Users\<usuario>\AppData\Local\Temp\ipykernel_26552\3248864415.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tr["text_len"] = tr["text"].str.len()
C:\Users\<usuario>\AppData\Local\Temp\ipykernel_26552\3248864415.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tr["n_words"] = tr["text"].str.split().str.len()



Balance de clases:
 label
1    1488
0    1350
Name: count, dtype: int64

Longitud (palabras) por clase:
        mean   50%    max
label                   
0      82.2  78.0  255.0
1      88.8  83.0  310.0

Posts por subreddit:
 subreddit
ptsd                584
relationships       552
anxiety             503
domesticviolence    316
assistance          289
survivorsofabuse    245
homeless            168
almosthomeless       80
stress               64
food_pantry          37
Name: count, dtype: int64


**Paso 4-T – Señal léxica.** Compara entre clases (estrés vs no estrés) el sentimiento medio del texto y las categorías LIWC — un diccionario psicolingüístico que mide qué % de palabras del post pertenece a categorías como *ansiedad*, *emoción negativa* o *primera persona*. Si difieren entre clases, el texto 'delata' el estrés.

La celda también genera y guarda la figura correspondiente (`plt.savefig`).

In [32]:
# PASO 4-T: Senal lexica por clase --------------------------------------------
print("\nSentimiento medio por clase:\n",
      tr.groupby("label")["sentiment"].mean().round(3))
liwc_vars = ["lex_liwc_anx", "lex_liwc_negemo", "lex_liwc_posemo", "lex_liwc_i"]
print("\nCategorias LIWC por clase (medias):")
print(tr.groupby("label")[liwc_vars].mean().round(2))

# Figura 3.1a - balance, sentimiento, LIWC ansiedad
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.countplot(data=tr, x="label", ax=axes[0], color="#937860")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["No estrés (0)", "Estrés (1)"])
axes[0].set_title(f"Balance de clases (train, n={len(tr)})")
sns.boxplot(data=tr, x="label", y="sentiment", ax=axes[1],
            hue="label", palette=["#55A868", "#C44E52"], legend=False)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["No estrés", "Estrés"])
axes[1].set_title("Sentimiento por clase")
sns.barplot(data=tr, x="label", y="lex_liwc_anx", ax=axes[2],
            hue="label", palette=["#55A868", "#C44E52"], legend=False)
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["No estrés", "Estrés"])
axes[2].set_title("Densidad de palabras de ansiedad (LIWC)\n2.5× mayor en la clase estrés")
plt.tight_layout()
plt.savefig(FIG + "b3_1_dreaddit_senales.png", dpi=130)
plt.close()

# Figura 3.1b - longitud y subreddits
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(data=tr, x="n_words", hue="label", bins=40, ax=axes[0],
             palette=["#55A868", "#C44E52"])
axes[0].set_title("Longitud de los posts (palabras) por clase")
axes[0].set_xlabel("Número de palabras")
tr["subreddit"].value_counts().sort_values().plot(kind="barh", ax=axes[1], color="#64B5CD")
axes[1].set_title("Posts por subreddit de origen")
plt.tight_layout()
plt.savefig(FIG + "b3_1_dreaddit_texto.png", dpi=130)
plt.close()


Sentimiento medio por clase:
 label
0    0.103
1   -0.016
Name: sentiment, dtype: float64

Categorias LIWC por clase (medias):
       lex_liwc_anx  lex_liwc_negemo  lex_liwc_posemo  lex_liwc_i
label                                                            
0              0.51             2.07             3.42        7.11
1              1.28             4.42             2.04       10.84


**Paso 5 – Hallazgos.** Síntesis de las conclusiones del dataset (se imprime tal cual para que quede registrada en la salida del notebook).

In [33]:
# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 3.1:
 - Clases casi balanceadas (1488 estres / 1350 no-estres): optimo para clasificar.
 - Sentimiento: +0.103 (no estres) vs -0.016 (estres).
 - Palabras de ansiedad LIWC: 1.28 vs 0.51 (x2.5 en clase estres).
 - Emocion negativa LIWC tambien discrimina; emocion positiva es mayor en no-estres.
 - Subreddits dominantes: ptsd, relationships, anxiety -> el estres capturado es
   relacional/clinico, no exclusivamente academico (considerar al generalizar).
 - 116 features precalculadas permiten comparar en el Objetivo 2 un modelo LIWC
   vs un modelo TF-IDF/embeddings propio.
""")


HALLAZGOS 3.1:
 - Clases casi balanceadas (1488 estres / 1350 no-estres): optimo para clasificar.
 - Sentimiento: +0.103 (no estres) vs -0.016 (estres).
 - Palabras de ansiedad LIWC: 1.28 vs 0.51 (x2.5 en clase estres).
 - Emocion negativa LIWC tambien discrimina; emocion positiva es mayor en no-estres.
 - Subreddits dominantes: ptsd, relationships, anxiety -> el estres capturado es
   relacional/clinico, no exclusivamente academico (considerar al generalizar).
 - 116 features precalculadas permiten comparar en el Objetivo 2 un modelo LIWC
   vs un modelo TF-IDF/embeddings propio.



## DATASET 3.2 — DEPRESSION STUDENT DATASET (prajwalg26, n=502)

In [34]:
print(SEP); print("DATASET 3.2 - DEPRESSION STUDENT (prajwalg26)"); print(SEP)

DATASET 3.2 - DEPRESSION STUDENT (prajwalg26)


**Paso 1 – Carga e inventario.** `pd.read_csv()` carga el archivo en un DataFrame (tabla en memoria); `.shape` muestra filas × columnas.

In [35]:
# PASO 1: Carga --------------------------------------------------------------
df2 = pd.read_csv(DATA + "3 Student_Mental_Health_Analysis GITHUB/"
                         "Depression+Student+Dataset.csv")
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")
print("NOTA METODOLOGICA: dataset tabular (sin texto). Complementa el bloque con "
      "un perfil estructurado del malestar estudiantil; el NLP recae en Dreaddit.")

Dimensiones: 502 filas x 11 columnas
NOTA METODOLOGICA: dataset tabular (sin texto). Complementa el bloque con un perfil estructurado del malestar estudiantil; el NLP recae en Dreaddit.


**Paso 2 – Calidad.** `.isna().sum()` cuenta valores vacíos por columna y `.duplicated().sum()` cuenta filas repetidas.

In [36]:
# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df2.isna().sum().sum(),
      "| Duplicados:", df2.duplicated().sum())

Nulos totales: 0 | Duplicados: 0


**Paso 3 – Univariante.** `.value_counts()` cuenta cuántos registros hay en cada nivel de la variable objetivo; `.describe()` resume media, desviación y extremos.

In [37]:
# PASO 3: Univariante --------------------------------------------------------
print("\nBalance del target Depression:\n", df2["Depression"].value_counts())


Balance del target Depression:
 Depression
Yes    252
No     250
Name: count, dtype: int64


**Paso 4 – Bivariante.** `.corr()` calcula la correlación de Pearson (-1 a +1) entre cada variable y la de estrés: cerca de 0 = sin relación lineal; |r| alto = relación fuerte. `pd.crosstab(..., normalize='index')` construye una tabla cruzada en porcentajes: qué % de cada grupo cae en cada categoría.

La celda también genera y guarda la figura correspondiente (`plt.savefig`).

In [38]:
# PASO 4: Bivariante ---------------------------------------------------------
ct = pd.crosstab(df2["Have you ever had suicidal thoughts ?"],
                 df2["Depression"], normalize="index") * 100
print("\n% Depresion segun pensamientos suicidas:\n", ct.round(1))
print("\nMedias por grupo de depresion:")
print(df2.groupby("Depression")[["Academic Pressure", "Study Satisfaction",
      "Financial Stress", "Study Hours"]].mean().round(2))

# Figura 3.2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ct["Yes"].plot(kind="bar", ax=axes[0], color=["#55A868", "#C44E52"])
axes[0].set_title("% con depresión según pensamientos suicidas")
axes[0].set_xlabel("¿Ha tenido pensamientos suicidas?")
axes[0].set_ylabel("% con depresión"); axes[0].tick_params(axis="x", rotation=0)
sns.boxplot(data=df2, x="Depression", y="Academic Pressure", ax=axes[1],
            hue="Depression", palette=["#55A868", "#C44E52"], legend=False)
axes[1].set_title("Presión académica según depresión")
plt.tight_layout()
plt.savefig(FIG + "b3_2_depression_student.png", dpi=130)
plt.close()


% Depresion segun pensamientos suicidas:
 Depression                               No   Yes
Have you ever had suicidal thoughts ?            
No                                     74.0  26.0
Yes                                    27.3  72.7

Medias por grupo de depresion:
            Academic Pressure  Study Satisfaction  Financial Stress  Study Hours
Depression                                                                      
No                       2.34                3.48              2.50         5.65
Yes                      3.66                2.67              3.35         7.15


**Paso 5 – Hallazgos.** Síntesis de las conclusiones del dataset (se imprime tal cual para que quede registrada en la salida del notebook).

In [39]:
# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 3.2:
 - 73% de quienes reportan pensamientos suicidas presentan depresion (vs 26%).
 - Presion academica: 3.66 (con depresion) vs 2.34 (sin) -> +57%.
 - Estres financiero tambien es mayor en el grupo con depresion.
 - Target balanceado (252/250): apto para clasificacion en el Objetivo 2.
""")

print(SEP)
print("EDA BLOQUE 3 COMPLETADO - 3 figuras generadas en", FIG)
print(SEP)


HALLAZGOS 3.2:
 - 73% de quienes reportan pensamientos suicidas presentan depresion (vs 26%).
 - Presion academica: 3.66 (con depresion) vs 2.34 (sin) -> +57%.
 - Estres financiero tambien es mayor en el grupo con depresion.
 - Target balanceado (252/250): apto para clasificacion en el Objetivo 2.

EDA BLOQUE 3 COMPLETADO - 3 figuras generadas en ..\salidas\objetivo1\figurasObje1\
